# Train Unified Graph-Mamba

All experiment parameters come from `configs/default.yaml`. The `run_name` field controls run-scoped paths: checkpoints go to `checkpoints/<run_name>/`, and graph/metrics outputs go to `output/<run_name>/`.

In [ ]:
from pathlib import Path
import copy
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'configs/default.yaml').exists():
    REPO_ROOT = Path('/mnt/data3/GraphMamba')

CONFIG_PATH = REPO_ROOT / 'configs/default.yaml'
sys.path.insert(0, str(REPO_ROOT / 'src'))

from graphmamba.config import load_config, resolve_run_paths

def make_repo_path(value):
    path = Path(value)
    return str(path if path.is_absolute() else REPO_ROOT / path)

config = copy.deepcopy(load_config(CONFIG_PATH))
config['paths']['checkpoint_root'] = make_repo_path(config['paths']['checkpoint_root'])
config['paths']['output_root'] = make_repo_path(config['paths']['output_root'])
run_paths = resolve_run_paths(config)

print(json.dumps({
    'config_path': str(CONFIG_PATH),
    'run_name': run_paths['run_name'],
    'date_range': [config['data']['start_date'], config['data']['end_date']],
    'lead_time': config['data'].get('lead_time', 0),
    'train_fraction': config['training']['train_fraction'],
    'val_fraction': config['training']['val_fraction'],
    'test_fraction': 1.0 - config['training']['train_fraction'] - config['training']['val_fraction'],
    'epochs': config['training']['epochs'],
    'batch_size': config['training']['batch_size'],
    'show_progress': config['training'].get('show_progress', True),
    'resume_training': config['training'].get('resume_training', False),
    'batch_log_interval': config['training'].get('batch_log_interval', 0),
    'device': config['training']['device'],
    'multi_gpu': config['training'].get('multi_gpu', 'off'),
    'gpu_min_free_memory_gb': config['training'].get('gpu_min_free_memory_gb', 0.0),
    'gpu_max_count': config['training'].get('gpu_max_count'),
    'checkpoint_dir': str(run_paths['checkpoint_dir']),
    'output_dir': str(run_paths['output_dir']),
    'graph_path': str(run_paths['graph_path']),
}, indent=2))

In [ ]:
from graphmamba.graph import build_unified_graph

graph_summary = build_unified_graph(
    tempo_dir=config['paths']['tempo_dir'],
    airnow_dir=config['paths']['airnow_dir'],
    graph_path=run_paths['graph_path'],
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date'],
    local_context_k=config['data']['local_context_k'],
    station_neighbor_k=config['data']['station_neighbor_k'],
    tempo_neighbor_k=config['data']['tempo_neighbor_k'],
)
graph_summary

In [ ]:
from graphmamba.data import TempoAirNowDataset, split_indices

dataset = TempoAirNowDataset(
    graph_path=run_paths['graph_path'],
    tempo_dir=config['paths']['tempo_dir'],
    airnow_dir=config['paths']['airnow_dir'],
    start_date=config['data']['start_date'],
    end_date=config['data']['end_date'],
    context_steps=config['data']['context_steps'],
    max_time_snap_hours=config['data']['max_time_snap_hours'],
    max_column=config['data']['max_column'],
    min_valid_targets=config['data']['min_valid_targets'],
    lead_time=config['data'].get('lead_time', 0),
    tempo_variable=config['data'].get('tempo_variable', 'product/vertical_column_troposphere'),
    tempo_quality_variable=config['data'].get('tempo_quality_variable', 'product/main_data_quality_flag'),
    tempo_cloud_variable=config['data'].get('tempo_cloud_variable', 'support_data/eff_cloud_fraction'),
    tempo_quality_flag_valid=config['data'].get('tempo_quality_flag_valid', 0),
    tempo_cloud_fraction_max=config['data'].get('tempo_cloud_fraction_max', 0.2),
)
train_idx, val_idx, test_idx = split_indices(
    len(dataset),
    config['training']['train_fraction'],
    config['training']['val_fraction'],
)
print(json.dumps({
    'aligned_windows': len(dataset),
    'lead_time': dataset.lead_time,
    'train_windows': int(len(train_idx)),
    'val_windows': int(len(val_idx)),
    'test_windows': int(len(test_idx)),
}, indent=2))

In [ ]:
from graphmamba.train import train

result = train(config)
print(json.dumps({
    'best_model': result['best_model'],
    'last_model': result['last_model'],
    'output_dir': result['output_dir'],
    'resume_training': result['resume_training'],
    'lead_time': result['lead_time'],
    'device': result['device'],
    'device_ids': result['device_ids'],
    'test': result['test'],
}, indent=2))

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(result['history'])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history['epoch'], history['train_loss'], marker='o', linewidth=1.8, label='Train loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss by Epoch')
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

print(f"Saved loss curve: {result['loss_plot']}")
history.tail()